# 03 — Modelagem

Estimação dos modelos de Poisson e Binomial Negativa com offset de exposição.

**Offset:** log(n_reviews) — controla pela exposição de cada vendedor, modelando a **taxa** de negativos.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial as NB_MLE
from scipy import stats
import json, warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/dataset_vendedores.csv")
print(f"Dataset: {len(df):,} vendedores")

Dataset: 1,754 vendedores


## 1. Preparação das variáveis

In [2]:
# Agrupar categorias com menos de 30 vendedores
cat_counts = df["categoria_principal"].value_counts()
cats_keep = cat_counts[cat_counts >= 30].index.tolist()
df["cat_group"] = df["categoria_principal"].where(df["categoria_principal"].isin(cats_keep), "other")
print(f"Categorias: {df['categoria_principal'].nunique()} -> {df['cat_group'].nunique()} agrupadas")
print(df["cat_group"].value_counts())

Categorias: 65 -> 19 agrupadas
cat_group
other                       318
sports_leisure              188
health_beauty               169
housewares                  163
furniture_decor             117
auto                        106
computers_accessories        96
bed_bath_table               85
toys                         79
baby                         56
garden_tools                 51
telephony                    48
pet_shop                     48
perfumery                    47
cool_stuff                   44
stationery                   39
watches_gifts                35
fashion_bags_accessories     35
electronics                  30
Name: count, dtype: int64


In [3]:
# Agrupar estados com menos de 30 vendedores
st_counts = df["seller_state"].value_counts()
st_keep = st_counts[st_counts >= 30].index.tolist()
df["state_group"] = df["seller_state"].where(df["seller_state"].isin(st_keep), "other")
print(f"Estados: {df['seller_state'].nunique()} -> {df['state_group'].nunique()} agrupados")
print(df["state_group"].value_counts())

Estados: 20 -> 7 agrupados
state_group
SP       1076
PR        190
MG        145
RJ         98
SC         95
other      92
RS         58
Name: count, dtype: int64


In [4]:
# Offset e dummies (referencia explicita: sports_leisure para categoria, SP para estado)
df["log_n_reviews"] = np.log(df["n_reviews"])

df_model = df[["n_negative", "log_n_reviews", "atraso_medio", "ticket_medio",
               "frete_medio", "peso_medio", "cat_group", "state_group"]].copy()
df_model = pd.get_dummies(df_model, columns=["cat_group", "state_group"], dtype=int)

# Remover dummies de referencia: sports_leisure (categoria base) e SP (estado base)
df_model = df_model.drop(columns=["cat_group_sports_leisure", "state_group_SP"])

y = df_model["n_negative"]
offset = df_model["log_n_reviews"]
X = df_model.drop(columns=["n_negative", "log_n_reviews"])
X = sm.add_constant(X)

print(f"Y: {y.shape}, X: {X.shape}")
print(f"Categoria de referencia: sports_leisure (n={(df["cat_group"]=="sports_leisure").sum()})")
print(f"Estado de referencia: SP (n={(df["state_group"]=="SP").sum()})")

Y: (1754,), X: (1754, 29)
Categoria de referencia: sports_leisure (n=188)
Estado de referencia: SP (n=1076)


## 2. Modelo de Poisson

In [5]:
poisson = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
print(poisson.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             n_negative   No. Observations:                 1754
Model:                            GLM   Df Residuals:                     1725
Model Family:                 Poisson   Df Model:                           28
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3715.3
Date:                Sun, 10 May 2026   Deviance:                       3002.8
Time:                        10:21:35   Pearson chi2:                 3.09e+03
No. Iterations:                     6   Pseudo R-squ. (CS):             0.2597
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

In [6]:
print(f"Log-Likelihood: {poisson.llf:.2f}")
print(f"AIC: {poisson.aic:.2f}")
print(f"BIC: {poisson.bic_llf:.2f}")
print(f"Deviance: {poisson.deviance:.2f}")
print(f"Pearson chi2/df: {poisson.pearson_chi2/poisson.df_resid:.4f}  (>>1 indica sobredispersao)")

Log-Likelihood: -3715.34
AIC: 7488.67
BIC: 7647.29
Deviance: 3002.79
Pearson chi2/df: 1.7887  (>>1 indica sobredispersao)


## 3. Teste de sobredispersão (Cameron-Trivedi)

- H0: Equidispersão (Poisson adequado)
- H1: Sobredispersão (BN necessária)

In [7]:
from statstests.tests import overdisp

print("=== Teste de superdispersao de Cameron-Trivedi (statstests) ===")
print("Pacote statstests: Favero, L.P. e Santos, H.P.")
print()
overdisp(poisson, df_model)

# Capturar valores para metricas (alinhado com a parametrizacao do statstests:
# regressao OLS sem intercepto de ((y-mu)^2 - y)/mu sobre mu)
mu_hat = poisson.fittedvalues
aux_dep = ((y - mu_hat)**2 - y) / mu_hat
aux_X = mu_hat.values.reshape(-1, 1)
aux_ols = sm.OLS(aux_dep, aux_X).fit()
alpha_est = aux_ols.params.iloc[0]
t_stat = aux_ols.tvalues.iloc[0]
p_val_ct = aux_ols.pvalues.iloc[0]
print(f"Valores capturados: alpha={alpha_est:.6f}, t={t_stat:.4f}, p={p_val_ct:.4e}")

=== Teste de superdispersao de Cameron-Trivedi (statstests) ===
Pacote statstests: Favero, L.P. e Santos, H.P.

Estimating overdispersion test for model type: GLMResultsWrapper...

                        Results: Ordinary least squares
Model:                  OLS              Adj. R-squared (uncentered): 0.035     
Dependent Variable:     y                AIC:                         10922.3440
Date:                   2026-05-10 10:21 BIC:                         10927.8137
No. Observations:       1754             Log-Likelihood:              -5460.2   
Df Model:               1                F-statistic:                 63.94     
Df Residuals:           1753             Prob (F-statistic):          2.31e-15  
R-squared (uncentered): 0.035            Scale:                       29.626    
--------------------------------------------------------------------------------------
             Coef.        Std.Err.         t          P>|t|        [0.025       0.975]
----------------------

## 4. Modelo Binomial Negativo

In [8]:
# Estimacao por MLE (NB2 com alpha estimado conjuntamente)
negbin = NB_MLE(y, X, exposure=np.exp(offset)).fit(method="bfgs", maxiter=500, disp=False)
print(negbin.summary())
alpha_full = negbin.params["alpha"]
print(f"Alpha MLE (modelo completo): {alpha_full:.6f}")

                     NegativeBinomial Regression Results                      
Dep. Variable:             n_negative   No. Observations:                 1754
Model:               NegativeBinomial   Df Residuals:                     1725
Method:                           MLE   Df Model:                           28
Date:                Sun, 10 May 2026   Pseudo R-squ.:                 0.02933
Time:                        10:21:35   Log-Likelihood:                -3469.3
converged:                       True   LL-Null:                       -3574.1
Covariance Type:            nonrobust   LLR p-value:                 1.016e-29
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                 -1.7887      0.073    -24.623      0.000      -1.931      -1.646
atraso_medio                           0.0407      0.004   


Alpha MLE (modelo completo): 0.113030


In [9]:
print(f"Log-Likelihood: {negbin.llf:.2f}")
print(f"AIC: {negbin.aic:.2f}")
print(f"BIC: {negbin.bic:.2f}")

Log-Likelihood: -3469.32
AIC: 6998.64
BIC: 7162.73


## 5. Comparação dos modelos

In [10]:
lr_stat = 2 * (negbin.llf - poisson.llf)
p_value_lr = stats.chi2.sf(abs(lr_stat), df=1)

print("=== Teste LR: Poisson vs NB2 MLE ===")
print(f"LR stat: {abs(lr_stat):.2f}")
print(f"p-valor: {p_value_lr:.2e}")

print(f"{'Metrica':<20s} {'Poisson':>15s} {'NB2 MLE':>15s} {'Melhor':>10s}")
print("-" * 63)
comparacoes = [
    ("Log-Likelihood", poisson.llf, negbin.llf, "maior"),
    ("AIC", poisson.aic, negbin.aic, "menor"),
    ("BIC", poisson.bic_llf, negbin.bic, "menor"),
]
for name, vp, vn, d in comparacoes:
    if d == "maior":
        m = "NB2 MLE" if vn > vp else "Poisson"
    else:
        m = "NB2 MLE" if vn < vp else "Poisson"
    print(f"{name:<20s} {vp:>15.2f} {vn:>15.2f} {m:>10s}")

=== Teste LR: Poisson vs NB2 MLE ===
LR stat: 492.03
p-valor: 5.15e-109
Metrica                      Poisson         NB2 MLE     Melhor
---------------------------------------------------------------
Log-Likelihood              -3715.34        -3469.32    NB2 MLE
AIC                          7488.67         6998.64    NB2 MLE
BIC                          7647.29         7162.73    NB2 MLE


## 6. Coeficientes e significância

In [11]:
coefs = pd.DataFrame({
    "coef": negbin.params, "std_err": negbin.bse,
    "z": negbin.tvalues, "p_value": negbin.pvalues,
    "exp_coef": np.exp(negbin.params)
})
coefs_reg = coefs.drop(index=["alpha"], errors="ignore")
coefs_reg["sig"] = coefs_reg["p_value"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
)

sig = coefs_reg[coefs_reg["p_value"] < 0.05]
nsig = coefs_reg[coefs_reg["p_value"] >= 0.05]

print(f"Significativas: {len(sig)}")
print(sig[["coef", "exp_coef", "p_value", "sig"]].to_string())
print(f"Nao significativas: {len(nsig)}")
if len(nsig) > 0:
    print(nsig[["coef", "exp_coef", "p_value"]].to_string())

Significativas: 7
                                     coef  exp_coef        p_value  sig
const                           -1.788690  0.167179  7.144541e-134  ***
atraso_medio                     0.040697  1.041537   1.029147e-20  ***
frete_medio                      0.003348  1.003354   3.836015e-02    *
cat_group_bed_bath_table         0.280580  1.323897   1.396884e-04  ***
cat_group_computers_accessories  0.212791  1.237126   4.721463e-03   **
cat_group_furniture_decor        0.254801  1.290205   5.690209e-04  ***
cat_group_telephony              0.325329  1.384487   5.372242e-04  ***
Nao significativas: 22
                                        coef  exp_coef   p_value
ticket_medio                        0.000116  1.000116  0.227003
peso_medio                          0.000012  1.000012  0.104518
cat_group_auto                      0.037705  1.038425  0.663813
cat_group_baby                     -0.027140  0.973225  0.782513
cat_group_cool_stuff               -0.128899  0.879062  0.

## 7. Modelo reduzido

In [12]:
vars_remover = [v for v in nsig.index.tolist() if v not in ["const", "alpha"]]

if len(vars_remover) > 0:
    print(f"Removendo {len(vars_remover)} variaveis nao significativas:")
    for v in vars_remover:
        pval = coefs.loc[v, "p_value"]
        print(f"  - {v} (p={pval:.4f})")
    X_red = X.drop(columns=vars_remover)
    negbin_red = NB_MLE(y, X_red, exposure=np.exp(offset)).fit(method="bfgs", maxiter=500, disp=False)
    n_red = X_red.shape[1]
    n_full = X.shape[1]
    alpha_red = negbin_red.params["alpha"]
    print(f"Modelo Reduzido: {n_red} vars (vs {n_full})")
    print(f"Alpha MLE reduzido: {alpha_red:.6f}")
    print(f"AIC: {negbin_red.aic:.2f} (vs {negbin.aic:.2f})")
    print(f"BIC: {negbin_red.bic:.2f} (vs {negbin.bic:.2f})")
    print(negbin_red.summary())
else:
    print("Todas significativas. Modelo reduzido = completo.")
    negbin_red = negbin
    X_red = X

Removendo 22 variaveis nao significativas:
  - ticket_medio (p=0.2270)
  - peso_medio (p=0.1045)
  - cat_group_auto (p=0.6638)
  - cat_group_baby (p=0.7825)
  - cat_group_cool_stuff (p=0.2528)
  - cat_group_electronics (p=0.9558)
  - cat_group_fashion_bags_accessories (p=0.2219)
  - cat_group_garden_tools (p=0.9464)
  - cat_group_health_beauty (p=0.9660)
  - cat_group_housewares (p=0.9808)
  - cat_group_other (p=0.7886)
  - cat_group_perfumery (p=0.3473)
  - cat_group_pet_shop (p=0.4389)
  - cat_group_stationery (p=0.3281)
  - cat_group_toys (p=0.5557)
  - cat_group_watches_gifts (p=0.0950)
  - state_group_MG (p=0.2037)
  - state_group_PR (p=0.5749)
  - state_group_RJ (p=0.8278)
  - state_group_RS (p=0.0844)
  - state_group_SC (p=0.9922)
  - state_group_other (p=0.4931)
Modelo Reduzido: 7 vars (vs 29)
Alpha MLE reduzido: 0.118401
AIC: 6976.66 (vs 6998.64)
BIC: 7020.41 (vs 7162.73)


                     NegativeBinomial Regression Results                      
Dep. Variable:             n_negative   No. Observations:                 1754
Model:               NegativeBinomial   Df Residuals:                     1747
Method:                           MLE   Df Model:                            6
Date:                Sun, 10 May 2026   Pseudo R-squ.:                 0.02625
Time:                        10:21:35   Log-Likelihood:                -3480.3
converged:                       True   LL-Null:                       -3574.1
Covariance Type:            nonrobust   LLR p-value:                 8.160e-38
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const                              -1.7802      0.056    -31.824      0.000      -1.890      -1.671
atraso_medio                        0.0435      0.004     10.641    

## 8. Salvar resultados

In [13]:
coefs_final = pd.DataFrame({
    "coef": negbin_red.params, "std_err": negbin_red.bse,
    "z": negbin_red.tvalues, "p_value": negbin_red.pvalues,
    "exp_coef": np.exp(negbin_red.params)
})
coefs_reg_final = coefs_final.drop(index=["alpha"], errors="ignore")
coefs_reg_final.to_csv("../outputs/coeficientes_negbin.csv")

# Pearson chi2/df para NB2 MLE
alpha_mle = negbin_red.params["alpha"]
mu_pred = negbin_red.predict(X_red, exposure=np.exp(offset))
pearson_resid2 = ((y - mu_pred)**2) / (mu_pred + alpha_mle * mu_pred**2)
pearson_chi2_nb = pearson_resid2.sum()
df_resid_nb = len(y) - (len(X_red.columns) + 1)
pearson_chi2_df_nb = pearson_chi2_nb / df_resid_nb
print(f"Pearson chi2/df (NB2 MLE): {pearson_chi2_df_nb:.4f}")

alpha_full_val = negbin.params["alpha"]
alpha_red_val = negbin_red.params["alpha"]
metricas = {
    "poisson": {
        "aic": float(poisson.aic), "bic": float(poisson.bic_llf),
        "llf": float(poisson.llf), "deviance": float(poisson.deviance),
        "pearson_chi2_df": float(poisson.pearson_chi2/poisson.df_resid),
        "n_vars": int(X.shape[1])},
    "negbin_completo": {
        "aic": float(negbin.aic), "bic": float(negbin.bic),
        "llf": float(negbin.llf), "n_vars": int(X.shape[1]),
        "alpha_mle": float(alpha_full_val)},
    "negbin_reduzido": {
        "aic": float(negbin_red.aic), "bic": float(negbin_red.bic),
        "llf": float(negbin_red.llf),
        "pearson_chi2_df": float(pearson_chi2_df_nb),
        "n_vars": int(X_red.shape[1]),
        "alpha_mle": float(alpha_red_val)},
    "cameron_trivedi": {
        "alpha": float(alpha_est), "t": float(t_stat), "p_value": float(p_val_ct)},
    "lr_test": {
        "statistic": float(abs(lr_stat)), "p_value": float(p_value_lr)}
}
with open("../outputs/metricas_modelos.json", "w") as f:
    json.dump(metricas, f, indent=2)
print("Salvos: coeficientes_negbin.csv e metricas_modelos.json")

Pearson chi2/df (NB2 MLE): 1.0506
Salvos: coeficientes_negbin.csv e metricas_modelos.json


## 9. Teste de Vuong — verificação de inflação de zeros

Embora o NB2 reduzido tenha apresentado bom ajuste (Pearson χ²/gl = 1,05), com 21% de zeros na amostra cabe verificar formalmente se um modelo Zero-Inflated Negative Binomial (ZINB) seria preferível.

- H0: não há inflação de zeros (NB2 adequado)
- H1: há inflação de zeros (ZINB preferível)

Referência: Vuong (1989); Fávero, Duarte e Santos (2024) — pacote `statstests`.

In [14]:
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
from statstests.tests import vuong_test

# Estimar ZINB com mesmas variaveis no count + intercepto no inflate
X_inflate = sm.add_constant(pd.DataFrame(index=range(len(y))))
zinb = ZeroInflatedNegativeBinomialP(
    y, X_red, exog_infl=X_inflate, offset=offset, inflation="logit"
).fit(method="bfgs", maxiter=500, disp=False)

ll_nb = negbin_red.llf
aic_nb = negbin_red.aic
ll_zinb = zinb.llf
aic_zinb = zinb.aic
delta = aic_zinb - aic_nb

print("=== Teste de Vuong: NB2 reduzido vs ZINB ===")
print(f"NB2  reduzido: LL = {ll_nb:.4f}, AIC = {aic_nb:.2f}")
print(f"ZINB         : LL = {ll_zinb:.4f}, AIC = {aic_zinb:.2f}")
print(f"Delta AIC (ZINB - NB2) = {delta:+.4f}")
print()
vuong_test(negbin_red, zinb)

metricas["vuong_test"] = {
    "nb2_aic": float(aic_nb), "zinb_aic": float(aic_zinb),
    "delta_aic": float(delta),
    "conclusao": "Vuong nao indicou superioridade estatistica do ZINB; AIC favorece NB2"
}
with open("../outputs/metricas_modelos.json", "w") as f:
    json.dump(metricas, f, indent=2)

=== Teste de Vuong: NB2 reduzido vs ZINB ===
NB2  reduzido: LL = -3480.3279, AIC = 6976.66
ZINB         : LL = -3480.3279, AIC = 6978.66
Delta AIC (ZINB - NB2) = +2.0000



Vuong Non-Nested Hypothesis Test-Statistic (Raw):
Vuong z-statistic: 0.0009959264269470529
p-value: 0.4996026829058028
